In [1]:
import sys

In [2]:
from typing import List, Dict, Any
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from neo4j import GraphDatabase

In [3]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain.schema import BaseRetriever
import re
from rapidfuzz import process

In [53]:
# Neo4j 연결 설정
NEO4J_URI = "bolt://neo4j-gds-apoc-n10s:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "neo4jpassword"

# Neo4j 연결 테스트
try:
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as test_driver:
        with test_driver.session() as session:
            result = session.run("MATCH (n) RETURN count(n) as count")
            count = result.single()["count"]
            print(f"Neo4j 연결 성공! 데이터베이스에 있는 노드 수: {count}")
except Exception as e:
    print(f"Neo4j 연결 실패: {str(e)}")

Neo4j 연결 성공! 데이터베이스에 있는 노드 수: 495


In [90]:
class ValidatedNeo4jGraph(Neo4jGraph):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cache_schema_elements()
    
    def _cache_schema_elements(self):
        """스키마 요소들을 캐시"""
        load_all_elements_query = """
        // 모든 노드 레이블 수집
        CALL() {
            MATCH (n)
            UNWIND labels(n) AS label
            RETURN 'NODE' AS entity_type, label, collect(DISTINCT keys(n)) AS key_collections
        }
        WITH entity_type, label, reduce(merged = [], key_list IN key_collections | merged + key_list) AS all_keys
        RETURN {
            entity_type: entity_type,
            label: label,
            properties: apoc.coll.toSet(all_keys)
        } AS schema_info

        UNION ALL

        // 모든 관계 타입 수집
        CALL() {
            MATCH ()-[r]->()
            WITH type(r) AS label, collect(DISTINCT keys(r)) AS key_collections
            RETURN 'RELATIONSHIP' AS entity_type, label, key_collections
        }
        WITH entity_type, label, reduce(merged = [], key_list IN key_collections | merged + key_list) AS all_keys
        RETURN {
            entity_type: entity_type,
            label: label,
            properties: apoc.coll.toSet(all_keys)
        } AS schema_info
        """
        # raw_schema = self.query(load_all_elements_query)
        raw_schema = super().query(load_all_elements_query)
        self.valid_labels = {item['schema_info']['label']:item['schema_info']['properties'] for item in raw_schema if item['schema_info']['entity_type'] == 'NODE'}
        self.valid_relationships = {item['schema_info']['label']:item['schema_info']['properties'] for item in raw_schema if item['schema_info']['entity_type'] == 'RELATIONSHIP'}
        
    def _extract_property_references(self, cypher_query: str):
        property_refs = []
        
        # 1. WHERE 절에서의 속성 참조 (예: n.name, person.age)
        where_pattern = r'(\w+)\.(\w+)\s*(?:[=<>!]+|CONTAINS|STARTS\s+WITH|ENDS\s+WITH|IN)'
        where_matches = re.findall(where_pattern, cypher_query, re.IGNORECASE)
        
        for var, prop in where_matches:
            label = self._get_variable_label(cypher_query, var)
            property_refs.append((var, label, prop))
        
        # 2. RETURN 절에서의 속성 참조
        return_pattern = r'RETURN.*?(\w+)\.(\w+)'
        return_matches = re.findall(return_pattern, cypher_query, re.IGNORECASE | re.DOTALL)
        
        for var, prop in return_matches:
            label = self._get_variable_label(cypher_query, var)
            property_refs.append((var, label, prop))
            
        # 3. ORDER BY 절에서의 속성 참조
        order_pattern = r'ORDER\s+BY\s+(\w+)\.(\w+)'
        order_matches = re.findall(order_pattern, cypher_query, re.IGNORECASE)
        
        for var, prop in order_matches:
            label = self._get_variable_label(cypher_query, var)
            property_refs.append((var, label, prop))
            
        # 4. MATCH 절에서 속성 필터 (n:Person {name: "John"})
        match_filter_pattern = r'(\w+):(\w+)\s*\{([^}]+)\}'
        match_filter_matches = re.findall(match_filter_pattern, cypher_query)
        
        for var, label, props_str in match_filter_matches:
            prop_pattern = r'(\w+)\s*:'
            props = re.findall(prop_pattern, props_str)
            for prop in props:
                property_refs.append((var, label, prop))
                
        return property_refs

        
    def _validate_cypher(self, cypher_query: str):
        """Cypher 쿼리의 스키마 요소들을 검증"""
        # 노드 레이블 검증
        label_pattern = r':(`?[0-9a-zA-Z가-힣]+`?) '
        found_labels = set(re.findall(label_pattern, cypher_query))
        invalid_labels = found_labels - set(self.valid_labels.keys())
        
        # 각 invalid label에 대해 가장 유사한 label을 매핑
        suggested_labels = {label: process.extractOne(label, self.valid_labels.keys()) for label in invalid_labels}
        
        # if invalid_labels:
        #     raise ValueError(f"Invalid node labels found: {invalid_labels}")
        
        # 관계 타입 검증 (예: -[:RELATIONSHIP_TYPE]->)
        rel_pattern = r'\[:(?[0-9a-zA-Z가-힣]+`?)\]'
        found_rels = set(re.findall(rel_pattern, cypher_query))
        invalid_rels = found_rels - set(self.valid_relationships.keys())
        
        # 각 invalid relationship에 대해 가장 유사한 관계 타입을 매핑
        suggested_rels = {rel: process.extractOne(rel, self.valid_relationships.keys()) for rel in invalid_rels}
        
        # if invalid_rels:
        #     raise ValueError(f"Invalid relationship types found: {invalid_rels}")
        
        # TODO: 속성 검증
    
    def query(self, query: str, params: dict = None, validation: bool = False):
        """쿼리 실행 전 검증"""
        if validation:
            self._validate_cypher(query)
        return super().query(query, params)

In [91]:
from langchain_community.graphs import Neo4jGraph

# Neo4jGraph 객체 생성
# graph = Neo4jGraph(
graph = ValidatedNeo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    enhanced_schema=True,
)

# schema = graph.get_schema
# print("Neo4j Schema:")
# print(schema)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.AggregationSkippedNull} {category: UNRECOGNIZED} {title: The query contains an aggregation function that skips null values.} {description: null value eliminated in set function.} {position: None} for query: 'MATCH (n:`개념`)\nWITH collect(distinct substring(toString(n.`설명`), 0, 50)) AS `설명_values`,\n     collect(disti

In [93]:
graph.get_structured_schema.keys()

dict_keys(['node_props', 'rel_props', 'relationships', 'metadata'])

In [96]:
graph.get_structured_schema.get("node_props").keys()


dict_keys(['요금제', '혜택', '음성통화', '문자메시지', '데이터용량', '충전서비스', '가입조건', '요금제그룹', '장애인공제', '개념', '라인업'])

In [97]:
graph.get_structured_schema.get("rel_props").keys()


dict_keys(['상품자동변경', '함께가입불가'])

In [98]:
graph.get_structured_schema.get("relationships")


[{'start': '요금제', 'type': '함께가입불가', 'end': '혜택'},
 {'start': '요금제', 'type': '제공', 'end': '충전서비스'},
 {'start': '요금제', 'type': '제공', 'end': '데이터용량'},
 {'start': '요금제', 'type': '제공', 'end': '문자메시지'},
 {'start': '요금제', 'type': '제공', 'end': '음성통화'},
 {'start': '요금제', 'type': '속함', 'end': '라인업'},
 {'start': '요금제', 'type': '속함', 'end': '요금제그룹'},
 {'start': '요금제', 'type': '제공혜택', 'end': '혜택'},
 {'start': '요금제', 'type': '가입조건', 'end': '가입조건'},
 {'start': '요금제', 'type': '장애인혜택', 'end': '장애인공제'},
 {'start': '요금제', 'type': '연관', 'end': '개념'},
 {'start': '요금제', 'type': '상품자동변경', 'end': '요금제'},
 {'start': '요금제', 'type': '데이터충전', 'end': '혜택'}]

In [99]:
graph.get_structured_schema.get("metadata").keys()

dict_keys(['constraint', 'index'])

In [87]:
# LLM 모델 초기화
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key="e97ee307-a791-4e06-ade1-df4b9d032eed",
    openai_api_base="https://aihub-api.sktelecom.com/aihub/v2/sandbox",
    streaming=True,
    temperature=0
)

# Neo4j 드라이버 초기화
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [88]:
sample_question = "5GX 프리미엄 요금제에 애플워치 무료로 사용할 수 있어?"

In [ ]:
from collections import namedtuple
import re
from collections import namedtuple
from typing import Any, Dict, List, Optional, Tuple

In [ ]:
Schema = namedtuple("Schema", ["left_node", "relation", "right_node"])

In [ ]:
corrector_schema = [
    Schema(el["start"], el["type"], el["end"])
    for el in graph.get_structured_schema.get("relationships", [])
]

In [ ]:
corrector_schema

In [ ]:
class CypherQueryCorrector:
    """
    Used to correct relationship direction in generated Cypher statements.
    This code is copied from the winner's submission to the Cypher competition:
    https://github.com/sakusaku-rich/cypher-direction-competition
    """

    property_pattern = re.compile(r"\{.+?\}")
    node_pattern = re.compile(r"\(.+?\)")
    path_pattern = re.compile(
        r"(\([^\,\(\)]*?(\{.+\})?[^\,\(\)]*?\))(<?-)(\[.*?\])?(->?)(\([^\,\(\)]*?(\{.+\})?[^\,\(\)]*?\))"
    )
    node_relation_node_pattern = re.compile(
        r"(\()+(?P<left_node>[^()]*?)\)(?P<relation>.*?)\((?P<right_node>[^()]*?)(\))+"
    )
    relation_type_pattern = re.compile(r":(?P<relation_type>.+?)?(\{.+\})?]")

    def __init__(self, schemas: List[Schema]):
        """
        Args:
            schemas: list of schemas
        """
        self.schemas = schemas

    def clean_node(self, node: str) -> str:
        """
        Args:
            node: node in string format

        """
        node = re.sub(self.property_pattern, "", node)
        node = node.replace("(", "")
        node = node.replace(")", "")
        node = node.strip()
        return node

    def detect_node_variables(self, query: str) -> Dict[str, List[str]]:
        """
        Args:
            query: cypher query
        """
        nodes = re.findall(self.node_pattern, query)
        nodes = [self.clean_node(node) for node in nodes]
        res: Dict[str, Any] = {}
        for node in nodes:
            parts = node.split(":")
            if parts == "":
                continue
            variable = parts[0]
            if variable not in res:
                res[variable] = []
            res[variable] += parts[1:]
        return res

    def extract_paths(self, query: str) -> "List[str]":
        """
        Args:
            query: cypher query
        """
        paths = []
        idx = 0
        while matched := self.path_pattern.findall(query[idx:]):
            matched = matched[0]
            matched = [
                m for i, m in enumerate(matched) if i not in [1, len(matched) - 1]
            ]
            path = "".join(matched)
            idx = query.find(path) + len(path) - len(matched[-1])
            paths.append(path)
        return paths

    def judge_direction(self, relation: str) -> str:
        """
        Args:
            relation: relation in string format
        """
        direction = "BIDIRECTIONAL"
        if relation[0] == "<":
            direction = "INCOMING"
        if relation[-1] == ">":
            direction = "OUTGOING"
        return direction

    def extract_node_variable(self, part: str) -> Optional[str]:
        """
        Args:
            part: node in string format
        """
        part = part.lstrip("(").rstrip(")")
        idx = part.find(":")
        if idx != -1:
            part = part[:idx]
        return None if part == "" else part

    def detect_labels(
        self, str_node: str, node_variable_dict: Dict[str, Any]
    ) -> List[str]:
        """
        Args:
            str_node: node in string format
            node_variable_dict: dictionary of node variables
        """
        splitted_node = str_node.split(":")
        variable = splitted_node[0]
        labels = []
        if variable in node_variable_dict:
            labels = node_variable_dict[variable]
        elif variable == "" and len(splitted_node) > 1:
            labels = splitted_node[1:]
        return labels

    def verify_schema(
        self,
        from_node_labels: List[str],
        relation_types: List[str],
        to_node_labels: List[str],
    ) -> bool:
        """
        Args:
            from_node_labels: labels of the from node
            relation_type: type of the relation
            to_node_labels: labels of the to node
        """
        valid_schemas = self.schemas
        if from_node_labels != []:
            from_node_labels = [label.strip("`") for label in from_node_labels]
            valid_schemas = [
                schema for schema in valid_schemas if schema[0] in from_node_labels
            ]
        if to_node_labels != []:
            to_node_labels = [label.strip("`") for label in to_node_labels]
            valid_schemas = [
                schema for schema in valid_schemas if schema[2] in to_node_labels
            ]
        if relation_types != []:
            relation_types = [type.strip("`") for type in relation_types]
            valid_schemas = [
                schema for schema in valid_schemas if schema[1] in relation_types
            ]
        return valid_schemas != []

    def detect_relation_types(self, str_relation: str) -> Tuple[str, List[str]]:
        """
        Args:
            str_relation: relation in string format
        """
        relation_direction = self.judge_direction(str_relation)
        relation_type = self.relation_type_pattern.search(str_relation)
        if relation_type is None or relation_type.group("relation_type") is None:
            return relation_direction, []
        relation_types = [
            t.strip().strip("!")
            for t in relation_type.group("relation_type").split("|")
        ]
        return relation_direction, relation_types

    def correct_query(self, query: str) -> str:
        """
        Args:
            query: cypher query
        """
        node_variable_dict = self.detect_node_variables(query)
        paths = self.extract_paths(query)
        for path in paths:
            original_path = path
            start_idx = 0
            while start_idx < len(path):
                match_res = re.match(self.node_relation_node_pattern, path[start_idx:])
                if match_res is None:
                    break
                start_idx += match_res.start()
                match_dict = match_res.groupdict()
                left_node_labels = self.detect_labels(
                    match_dict["left_node"], node_variable_dict
                )
                right_node_labels = self.detect_labels(
                    match_dict["right_node"], node_variable_dict
                )
                end_idx = (
                    start_idx
                    + 4
                    + len(match_dict["left_node"])
                    + len(match_dict["relation"])
                    + len(match_dict["right_node"])
                )
                original_partial_path = original_path[start_idx : end_idx + 1]
                relation_direction, relation_types = self.detect_relation_types(
                    match_dict["relation"]
                )
                print(original_partial_path)
                print(relation_direction)
                print(relation_types)

                if relation_types != [] and "".join(relation_types).find("*") != -1:
                    start_idx += (
                        len(match_dict["left_node"]) + len(match_dict["relation"]) + 2
                    )
                    continue

                if relation_direction == "OUTGOING":
                    is_legal = self.verify_schema(
                        left_node_labels, relation_types, right_node_labels
                    )
                    if not is_legal:
                        is_legal = self.verify_schema(
                            right_node_labels, relation_types, left_node_labels
                        )
                        print(is_legal) 
                        if is_legal:
                            corrected_relation = "<" + match_dict["relation"][:-1]
                            corrected_partial_path = original_partial_path.replace(
                                match_dict["relation"], corrected_relation
                            )
                            query = query.replace(
                                original_partial_path, corrected_partial_path
                            )
                        else:
                            return ""
                elif relation_direction == "INCOMING":
                    is_legal = self.verify_schema(
                        right_node_labels, relation_types, left_node_labels
                    )
                    if not is_legal:
                        is_legal = self.verify_schema(
                            left_node_labels, relation_types, right_node_labels
                        )
                        if is_legal:
                            corrected_relation = match_dict["relation"][1:] + ">"
                            corrected_partial_path = original_partial_path.replace(
                                match_dict["relation"], corrected_relation
                            )
                            query = query.replace(
                                original_partial_path, corrected_partial_path
                            )
                        else:
                            return ""
                else:
                    is_legal = self.verify_schema(
                        left_node_labels, relation_types, right_node_labels
                    )
                    is_legal |= self.verify_schema(
                        right_node_labels, relation_types, left_node_labels
                    )
                    if not is_legal:
                        return ""

                start_idx += (
                    len(match_dict["left_node"]) + len(match_dict["relation"]) + 2
                )
        return query

    def __call__(self, query: str) -> str:
        """Correct the query to make it valid. If
        Args:
            query: cypher query
        """
        return self.correct_query(query)

In [ ]:
cypher_query_corrector = CypherQueryCorrector(corrector_schema)

In [ ]:
cypher_query_corrector("MATCH (p:`요금제`)-[:`개념`]->(c:`개념`) WHERE c.`키워드` = '실버요금제' RETURN p, c")

In [ ]:
print("Human: Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided in the schema.\nKorean terms should be surrounded by backticks (``).\n\nSchema:\nNode properties:\n- **요금제**\n  - `고유ID`: STRING Example: \"PA00000091\"\n  - `라인업`: STRING Example: \"0청년 다이렉트플랜\"\n  - `청구방법`: STRING Available options: ['후불']\n  - `마케팅키워드`: LIST Min Size: 1, Max Size: 39\n  - `상품설명`: STRING Example: \"무제한 데이터와 0청년 특화 혜택을 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서\"\n  - `월정액`: INTEGER Min: 8800, Max: 125000\n  - `부가세제외월정액`: INTEGER Min: 8000, Max: 113637\n  - `net가격`: INTEGER Min: 8000, Max: 113637\n  - `상품가입조건`: STRING Example: \"T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 만 34세 이하 고객 가입 가\"\n  - `운영상태`: STRING Available options: ['운영', '가입중단']\n  - `상품분류`: STRING Available options: ['상품 > 기본요금제 > 휴대폰 요금제', '상품 > 기본요금제']\n  - `선택약정할인포함부가세제외월정액`: INTEGER Min: 7425, Max: 125000\n  - `상품코드매핑`: LIST Min Size: 1, Max Size: 1\n  - `영문상품명`: STRING Example: \"0 Youth direct 62\"\n  - `상품명`: STRING Example: \"0 청년 다이렉트 62\"\n- **혜택**\n  - `할인양`: STRING Example: \"7900원\"\n  - `혜택유형`: STRING Available options: ['이용료 할인', '요금 할인', '서비스 콘텐츠 할인']\n  - `할인유형`: STRING Available options: ['정액할인', '정률할인', '']\n  - `혜택명`: STRING Example: \"FLO 무료\"\n  - `최대할인금액`: STRING Example: \"7900원\"\n  - `마케팅키워드`: LIST Min Size: 4, Max Size: 36\n  - `혜택ID`: STRING Example: \"BA00000038\"\n- **음성통화**\n  - `리필비율한도`: FLOAT Min: null, Max: 0.2\n  - `지정번호통화제공량`: STRING Available options: ['null', '2.0']\n  - `음성통화제공량`: INTEGER Min: 0, Max: 99999\n  - `영상및부가통화제공량`: INTEGER Min: 0, Max: 400\n- **문자메시지**\n  - `문자제공량`: INTEGER Min: 50, Max: 99999\n- **데이터용량**\n  - `기본제공데이터용량`: FLOAT Min: 0.0, Max: 99999.0\n  - `기본제공데이터중공유및테더링가능용량`: FLOAT Min: 0.0, Max: 120.0\n  - `데이터소진후최대금액및속도제한적용`: STRING Available options: ['N', 'Y']\n  - `데이터리필쿠폰선물가능여부`: STRING Available options: ['Y', 'null']\n  - `데이터선물받기가능여부`: STRING Available options: ['Y', 'null']\n  - `시니어대상데이터소진후최대금액및속도제한적용`: STRING Available options: ['N', 'Y']\n  - `기본제공데이터중mvoip용량`: FLOAT Min: 0.0, Max: 99999.0\n  - `데이터리필가능용량`: FLOAT Min: 0.29296875, Max: 99999.0\n  - `데이터소진후데이터제공속도`: FLOAT Min: 0.0, Max: 5.0\n  - `최대데이터선물가능용량`: FLOAT Min: 0.0, Max: 2.0\n- **충전서비스**\n  - `최대충전금액`: FLOAT Min: 0.0, Max: 20000.0\n  - `최소충전금액`: FLOAT Min: 0.0, Max: 1000.0\n  - `충전서비스대상여부`: STRING Available options: ['N', 'Y']\n- **가입조건**\n  - `고객유형별가입가능여부`: STRING Available options: ['null', 'true', 'false']\n  - `선택약정동시가입가능여부`: STRING Available options: ['Y', 'null']\n  - `다이렉트플랜가입가능여부`: STRING Available options: ['Y', 'null']\n  - `가입가능최대나이`: INTEGER Min: 12, Max: 999\n  - `최대나이계산기준`: STRING Available options: ['null', '월기준']\n  - `T지원금약정동시가입가능여부`: STRING Available options: ['Y', 'null']\n  - `가입가능최소나이계산기준`: STRING Available options: ['null', '일기준', '월기준']\n  - `군인전용요금제여부`: STRING Available options: ['null', 'Y']\n  - `가입가능최소나이`: INTEGER Min: 0, Max: 80\n  - `동일명의가입가능여부`: STRING Available options: ['null', 'false']\n  - `개인고객세부유형별가입가능여부`: STRING Available options: ['null', 'true']\n- **요금제그룹**\n  - `그룹명`: STRING Available options: ['TING_PRCPLN', 'ONESVC_SILVER_PROD', 'NETFLIX_PRCPLN', '디즈니+ 요금제', '스마트기기 요금제', '유튜프 프리미엄 요금제']\n- **장애인공제**\n  - `장애인부가통화추가제공량`: INTEGER Min: 0, Max: 250\n- **개념**\n  - `설명`: STRING Available options: ['Plans for customers aged 65 and older', 'Pay-as-you-go plans', 'Plans that offer music streaming services', 'Military plans are not available to professional s', 'Price discounts means that the discount is applied']\n  - `키워드`: STRING Available options: ['실버요금제', '시니어요금제', '종량제요금제', '음악듣기요금제', '장교', '요금 할인']\n  - `CYPHER_TEMPLATE`: STRING Available options: ['MATCH (p:`요금제`)-[:`가입조건`]->(a:`가입조건`) WHERE a.`가입가', 'MATCH (p:`요금제`)-[:`제공`]->(d:`데이터용량`) WHERE d.`기본제공', 'MATCH (p:요금제)-[:속함]->(:요금제그룹 {그룹명: \"유튜브 프리미엄 요금제\"}']\nRelationship properties:\n- **상품자동변경**\n  - `기준일`: STRING Available options: ['만36세', '제대일', '만20세', '만14세', '넷플릭스 가입일']\n  - `변경일`: STRING Available options: ['기준일 + 1개월 1일', '기준일 + 8일']\nThe relationships:\n(:요금제)-[:제공]->(:충전서비스)\n(:요금제)-[:제공]->(:데이터용량)\n(:요금제)-[:제공]->(:문자메시지)\n(:요금제)-[:제공]->(:음성통화)\n(:요금제)-[:속함]->(:요금제그룹)\n(:요금제)-[:제공혜택]->(:혜택)\n(:요금제)-[:상품자동변경]->(:요금제)\n(:요금제)-[:가입조건]->(:가입조건)\n(:요금제)-[:장애인혜택]->(:장애인공제)\n(:개념)-[:연관]->(:요금제)\n\nDomain mapping and other rules:\n- For 기본제공데이터용량 (data limit), 문자제공량 (sms limit), 음성통화제공량 (voice limit) and other similar numeric fields about capacity, treat the term 무제한 (unlimited) as the value 99999. Do not apply this rule to price fields.\n- For questions about cheap or expensive plans, sort by the value of 월정액 (monthly price).\n- For search keywords, prefer a single noun split by a space. For example, use \"넷플릭스\" instead of \"넷플릭스 할인\".\n- For comparing products, generate a Cypher query that retrieves all products to be compared, and then compare the results.\n\nFor age-related queries, generate WHERE clause based on the following examples:\n- Plans only for 18 years old -> 가입가능최대나이 = 18 AND 가입가능최소나이 = 18\n- Plans for 18 years old -> 가입가능최대나이 >= 18 AND 가입가능최소나이 <= 18\n- Plans only for 18 years old and above -> 가입가능최대나이 >= 18 AND 가입가능최소나이 <= 18\n- Plans only for 18 years old and below -> 가입가능최대나이 <= 18 AND 가입가능최소나이 <= 18\n- Plans only for younger than 13 years old -> 가입가능최대나이 < 13 AND 가입가능최소나이 < 13\n\nFor querying list properties, do not use the CONTAINS operator directly on the array itself.\nInstead, use one of the following methods depending on the query intent:\n- To check for exact inclusion of a value: 'value' IN node.array_property\n- To check if any element partially matches a condition (e.g., substring): ANY(item IN node.array_property WHERE item CONTAINS 'value')\n- To check if all elements satisfy a condition: ALL(item IN node.array_property WHERE item CONTAINS 'value')\n\nExample:\n- \"Find plans that are only available for under 18\" -> \"MATCH (p:`요금제`) WHERE p.`가입가능최대나이` < 18 AND p.`가입가능최소나이` < 18 RETURN p\"\n- \"Compare 5GX 프리미엄 plan with other plans that have similar price\" -> \"MATCH (p:`요금제` {`상품명`: '5GX 프리미엄'}) WITH p, p.`월정액` AS reference_price  MATCH (other:`요금제`) WHERE ABS(other.`월정액` - reference_price) <= reference_price * 0.1 RETURN p AS `기준상품`, other AS `유사상품` ORDER BY ABS(other.`월정액` - reference_price)\"\n- \"Find one unlimited data plan\" -> \"MATCH (p:`요금제`) WHERE p.`기본제공데이터용량` = 99999 RETURN p LIMIT 1\"\n- \"Find discount benefits for 65 and above\" -> \"MATCH (p:`요금제`)-[:`가입조건`]->(c:`가입조건`) WHERE c.`가입가능최소나이` >= 65 RETURN p, c\"\n\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\nInclude the nodes and properties related to the question in the result.\n\nThe question is:\n실버 요금제")

In [ ]:
# 상태 타입 정의
class State:
    def __init__(self):
        self.messages: List = []
        self.steps: List[str] = []
        self.current_step: int = 0
        self.results: Dict = {}
        self.final_answer: Dict = {}

# 질문 분석 및 스텝 생성 함수
def analyze_question(state: State) -> State:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "사용자의 모바일 요금제 관련 질문을 분석하여 필요한 검색 단계들을 생성하세요."),
        MessagesPlaceholder(variable_name="messages"),
    ])
    
    response = llm.invoke(prompt.format_messages(messages=state.messages))
    steps = response.content.split('\n')
    
    state.steps = steps
    return state

In [ ]:
# 테스트를 위한 상태 객체 생성
state = State()
state.messages = [
    HumanMessage(content="가장 인기 많은 무제한 요금제 알려줘")
]

# analyze_question 함수 테스트
try:
    result_state = analyze_question(state)
    print("분석된 검색 단계:")
    for i, step in enumerate(result_state.steps):
        print(f"{i+1}. {step}")
except Exception as e:
    print(f"에러 발생: {str(e)}")


In [ ]:

# Cypher 쿼리 생성 및 실행 함수
def execute_step(state: State) -> State:
    current_step = state.steps[state.current_step]
    
    # Cypher 쿼리 생성
    prompt = ChatPromptTemplate.from_messages([
        ("system", "주어진 검색 단계에 대한 Neo4j Cypher 쿼리를 생성하세요."),
        ("user", f"단계: {current_step}")
    ])
    
    query_response = llm.invoke(prompt.format_messages())
    cypher_query = query_response.content
    
    # Neo4j 쿼리 실행
    with driver.session() as session:
        result = session.run(cypher_query).data()
        state.results[state.current_step] = result
    
    return state

# 결과 검증 함수
def validate_result(state: State) -> Dict[str, Any]:
    current_step = state.steps[state.current_step]
    current_result = state.results[state.current_step]
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "현재 단계의 검색 결과가 적절한지 검증하세요."),
        ("user", f"단계: {current_step}\n결과: {current_result}")
    ])
    
    validation = llm.invoke(prompt.format_messages())
    is_valid = "유효함" in validation.content
    
    if is_valid:
        if state.current_step < len(state.steps) - 1:
            state.current_step += 1
            return {"next": "execute_step"}
        else:
            return {"next": "final_answer"}
    else:
        return {"next": "execute_step"}

# 최종 답변 생성 함수
def generate_final_answer(state: State) -> State:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "모든 검색 결과를 종합하여 최종 답변을 JSON 형식으로 생성하세요."),
        ("user", f"검색 결과: {state.results}")
    ])
    
    response = llm.invoke(prompt.format_messages())
    state.final_answer = eval(response.content)
    return state

# 워크플로우 그래프 생성
workflow = StateGraph(State)

# 노드 추가
workflow.add_node("analyze_question", analyze_question)
workflow.add_node("execute_step", execute_step)
workflow.add_node("validate_result", validate_result)
workflow.add_node("final_answer", generate_final_answer)

# 엣지 연결
workflow.set_entry_point("analyze_question")
workflow.add_edge("analyze_question", "execute_step")
workflow.add_edge("execute_step", "validate_result")
workflow.add_conditional_edges(
    "validate_result",
    {
        "execute_step": lambda x: x["next"] == "execute_step",
        "final_answer": lambda x: x["next"] == "final_answer"
    }
)
workflow.add_edge("final_answer", END)

# 그래프 컴파일
app = workflow.compile()

# 사용 예시
def search_mobile_plans(question: str) -> Dict:
    state = State()
    state.messages = [HumanMessage(content=question)]
    result = app.invoke(state)
    return result.final_answer
